In [ ]:
!pip install -q langchain-text-splitters

In [1]:
# Cài đặt các thư viện cốt lõi cho E5 và xử lý dữ liệu
!pip install -q sentence-transformers pandas pyarrow tqdm torch

In [6]:
import re
import json
import os
import torch
import traceback
import numpy as np
import pandas as pd
# import open_clip
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & HẰNG SỐ (Đã sửa để chạy tiếp)
# ==========================================
INPUT_JSON = '/kaggle/input/datasets/baubauuu/toighetdatamining/merged_data_final.json'
OUTPUT_DIR = '/kaggle/working/Milvus_Export_Data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ĐƯỜNG DẪN FILE LOG ÔNG VỪA UPLOAD (INPUT)
INPUT_LOG = '/kaggle/input/datasets/baubauuu/logiddalay1/processed_ids.txt'

# FILE LOG ĐỂ GHI TIẾP (OUTPUT)
PROGRESS_FILE  = os.path.join(OUTPUT_DIR, 'processed_ids.txt')

# CÁC FILE PARQUET ĐỂ LƯU DỮ LIỆU MỚI (PART 2)
OUT_SUM_PQ     = os.path.join(OUTPUT_DIR, 'milvus_summary.parquet')
# OUT_CHK_PQ     = os.path.join(OUTPUT_DIR, 'milvus_chunks.parquet')
ERROR_LOG_FILE = os.path.join(OUTPUT_DIR, 'error_log.txt')

# --- TỰ ĐỘNG NẠP LOG CŨ VÀO WORKING ---
if os.path.exists(INPUT_LOG):
    import shutil
    # Copy từ Input sang Working để code cũ của ông đọc và ghi tiếp được
    shutil.copy(INPUT_LOG, PROGRESS_FILE)
    print(f"Đã nạp log cũ thành công! File log hiện tại có: {os.path.getsize(PROGRESS_FILE)} bytes")
else:
    print(" Không tìm thấy file log tại đường dẫn Input. Check lại tên Dataset nhé!")

# DIM_CLIP = 1024 
DIM_E5   = 1024 
BATCH_SIZE = 300

Đã nạp log cũ thành công! File log hiện tại có: 0 bytes


In [ ]:
# 2. KHỞI TẠO MODEL & DEVICE
# ==========================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f" Thiết bị đang sử dụng: {device.upper()}")


# # Load E5
# print(" Đang nạp Model E5 (Multilingual-Large)...")
# model_text = SentenceTransformer('intfloat/multilingual-e5-large', device=device)

# 2. KHỞI TẠO MODEL E5-LARGE-V2
# ==========================================
print(" Đang nạp Model E5 (English-Large-V2)...")
model_text = SentenceTransformer('intfloat/e5-large-v2', device=device)

 Thiết bị đang sử dụng: CUDA
 Đang nạp Model E5 (Multilingual-Large)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 3. CÁC HÀM HỖ TRỢ
# ==========================================
def load_processed_ids():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return set(line.strip() for line in f)
    return set()

def is_valid_vec(vec, expected_dim):
    if vec is None: return False
    v = np.array(vec)
    if v.size == 0 or v.shape[-1] != expected_dim: return False
    if np.isnan(v).any() or np.isinf(v).any(): return False
    return True

def clean_txt(text):
    if not text: return ""
    text = re.sub(r'\[\d+\]|\[citation needed\]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def enrich_and_split(item, splitter):
    title = item.get("title", "Unknown")
    full_text = item.get("full_text", "")
    if not full_text or len(full_text.strip()) < 50: return []

    parts = re.split(r'(==+.*?==+)', full_text)
    current_header = "General Information"
    final_chunks = []

    for part in parts:
        part = part.strip()
        if not part: continue
        if re.match(r'==+.*?==+', part):
            current_header = part.strip('= ').strip()
            continue
        
        blacklist = ['references', 'see also', 'external links', 'notes']
        if current_header.lower() in blacklist: continue

        clean_part = clean_txt(part)
        if len(clean_part) < 20: continue

        sub_chunks = splitter.split_text(clean_part)
        for c in sub_chunks:
            # Gắn context cho chunk để search chính xác hơn
            enriched = f"[{title} - {current_header}] {c}"
            final_chunks.append(enriched)
    return final_chunks

def save_append_parquet(new_data, file_path):
    if not new_data: return
    df_new = pd.DataFrame(new_data)
    if os.path.exists(file_path):
        df_old = pd.read_parquet(file_path)
        pd.concat([df_old, df_new], ignore_index=True).to_parquet(file_path, index=False)
    else:
        df_new.to_parquet(file_path, index=False)

In [ ]:
# ==========================================
# 4. QUY TRÌNH XỬ LÝ CHÍNH (E5 SUMMARY + E5 CHUNKS)
# ==========================================
def process_and_export():
    splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)
    
    print(f" Đang mở file {INPUT_JSON}...")
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        data = json.load(f)

    processed_ids = load_processed_ids()
    items_to_process = [item for item in data if str(item.get("id_dia_diem")) not in processed_ids]
    
    if not items_to_process:
        print(" Tuyệt vời! Dữ liệu đã được xử lý hoàn tất từ trước.")
        return

    print(f" Bắt đầu xử lý {len(items_to_process)} địa điểm mới với E5...")
    
    batch_summaries = []
    batch_chunks = []
    count_summary, count_chunk = 0, 0

    pbar = tqdm(items_to_process, desc="Mining Wikipedia")

    for i, item in enumerate(pbar):
        idx = str(item.get("id_dia_diem"))
        title = item.get("title", "Unknown")

        try:
            # --- 1. E5 Summary (Đã chuyển từ CLIP sang E5) ---
            raw_summary = item.get('summary_text', '')
            clean_summary = clean_txt(raw_summary)
            
            if clean_summary:
                # Định dạng prefix "passage: " bắt buộc cho dòng mô hình E5
                text_sum_e5 = f"passage: {title}: {clean_summary}"
                
                with torch.no_grad():
                    s_vec = model_text.encode([text_sum_e5], show_progress_bar=False)[0]

                # Kiểm tra vector hợp lệ với số chiều DIM_E5 (1024)
                if is_valid_vec(s_vec, DIM_E5):
                    batch_summaries.append({
                        "id": int(idx), 
                        "vector": s_vec.tolist(), 
                        "title": title,
                        "content": clean_summary  # THÊM TÓM TẮT VÀO METADATA THEO YÊU CẦU
                    })
                    count_summary += 1

            # --- 2. E5 Chunks (Giữ nguyên logic cũ) ---
            chunks = enrich_and_split(item, splitter)
            if chunks:
                texts_e5 = ["passage: " + clean_txt(c) for c in chunks]
                # Xử lý batch nhỏ bên trong để tránh nổ VRAM
                c_vecs = model_text.encode(texts_e5, batch_size=32, show_progress_bar=False)

                for j, (txt, v) in enumerate(zip(chunks, c_vecs)):
                    if is_valid_vec(v, DIM_E5):
                        batch_chunks.append({
                            "chunk_id": f"{idx}_{j}",
                            "parent_id": int(idx),
                            "vector": v.tolist(),
                            "content": txt
                        })
                        count_chunk += 1

            pbar.set_postfix({"Sum": count_summary, "Chunks": count_chunk})

            # --- 3. Lưu dữ liệu theo đợt ---
            if (len(batch_summaries) >= BATCH_SIZE) or (i == len(items_to_process) - 1):
                pbar.set_description(" Saving Parquet")
                
                if batch_summaries:
                    save_append_parquet(batch_summaries, OUT_SUM_PQ)
                if batch_chunks:
                    save_append_parquet(batch_chunks, OUT_CHK_PQ)
                
                print(f"  {len(batch_summaries)} địa điểm vào file. Tổng cộng Sum: {count_summary}")
                
                # Cập nhật checkpoint
                with open(PROGRESS_FILE, 'a') as f_p:
                    for s_item in batch_summaries:
                        f_p.write(str(s_item['id']) + "\n")
                
                batch_summaries, batch_chunks = [], []
                pbar.set_description("Mining Wikipedia")

        except Exception as e:
            with open(ERROR_LOG_FILE, 'a', encoding='utf-8') as f_err:
                f_err.write(f"Lỗi tại ID {idx} ({title}): {str(e)}\n{traceback.format_exc()}\n")
            continue

    print(f"\n" + "="*40)
    print(f" HOÀN TẤT QUY TRÌNH!")
    print(f" Summary vectors: {count_summary}")
    print(f" Chunk vectors:   {count_chunk}")
    print(f" Kết quả tại: {OUTPUT_DIR}")
    print("="*40)

# Chạy trực tiếp
process_and_export()

  Đang mở file nguồn /kaggle/input/datasets/baubauuu/toighetdatamining/merged_data_final.json...
  Bắt đầu tạo vector E5 cho 130708 địa điểm...


E5 Summary Mining:   0%|          | 0/130708 [00:00<?, ?it/s]


  HOÀN TẤT QUY TRÌNH!
  Tổng số Summary (E5): 130655
  File kết quả: /kaggle/working/Milvus_Export_Data/milvus_summary.parquet
